In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime, timezone
import uuid

# Pipeline configuration
BRONZE_TABLE = "workspace.default.sentinel_bronze_inventory"

VOLUME_PATH = "/Volumes/workspace/default/sentinel_volume/inventory/"

INGESTION_RUN_ID = str(uuid.uuid4())

print("Smart Data Quality Sentinel - Ingestion Started")
print("Ingestion Run ID:", INGESTION_RUN_ID)
print("Volume Path:", VOLUME_PATH)

Smart Data Quality Sentinel - Ingestion Started
Ingestion Run ID: 93c661c2-cf54-434f-9de9-651ab35a882e
Volume Path: /Volumes/workspace/default/sentinel_volume/inventory/


sentinel_files/
- └── inventory/
- ├── incoming/
- ├── processed/
- └── failed/

In [0]:
BASE_PATH = "/Volumes/workspace/default/sentinel_files"

folders = [
    f"{BASE_PATH}/inventory/incoming",
    f"{BASE_PATH}/inventory/processed",
    f"{BASE_PATH}/inventory/failed"
]

for folder in folders:
    dbutils.fs.mkdirs(folder)
    print("Created:", folder)

Created: /Volumes/workspace/default/sentinel_files/inventory/incoming
Created: /Volumes/workspace/default/sentinel_files/inventory/processed
Created: /Volumes/workspace/default/sentinel_files/inventory/failed


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.sentinel_file_tracking (
    file_name STRING,
    file_path STRING,
    file_modification_time TIMESTAMP,
    ingestion_timestamp TIMESTAMP,
    records_ingested BIGINT,
    ingestion_status STRING,
    ingestion_run_id STRING
)
USING DELTA;

In [0]:
BASE_PATH = "/Volumes/workspace/default/sentinel_files"

INCOMING_PATH = f"{BASE_PATH}/inventory/incoming"
PROCESSED_PATH = f"{BASE_PATH}/inventory/processed"
FAILED_PATH = f"{BASE_PATH}/inventory/failed"

TRACKING_TABLE = "workspace.default.sentinel_file_tracking"

# Inspect incoming directory
incoming_files = dbutils.fs.ls(INCOMING_PATH)

print("Files available for ingestion:")

for file in incoming_files:
    if not file.isDir():
        print(
            "Name:", file.name,
            "| Size:", file.size,
            "| Modified:", file.modificationTime
        )

Files available for ingestion:
Name: sentinel_idempotency_test(processing_key_implementation_test).csv | Size: 198 | Modified: 1793476633000


## File discovery and ingestion

In [0]:
EXPECTED_COLUMNS = [
    "record_id",
    "product_id",
    "product_name",
    "category",
    "supplier_id",
    "cost_price",
    "selling_price",
    "quantity",
    "reorder_level",
    "last_updated"
]

print("Expected inventory columns:", EXPECTED_COLUMNS)

Expected inventory columns: ['record_id', 'product_id', 'product_name', 'category', 'supplier_id', 'cost_price', 'selling_price', 'quantity', 'reorder_level', 'last_updated']


In [0]:
import hashlib

def calculate_file_hash(file_path):

    sha256 = hashlib.sha256()

    local_path = file_path.replace(
        "dbfs:/Volumes/",
        "/Volumes/"
    )

    with open(local_path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024), b""
        ):
            sha256.update(chunk)

    return sha256.hexdigest()

In [0]:
# FUNC: ingest_inventory_file 
def ingest_inventory_file(file_info):

    file_name = file_info.name
    file_path = file_info.path
    file_modified_time = datetime.fromtimestamp(
        file_info.modificationTime / 1000,
        timezone.utc
    )

    destination_path = f"{PROCESSED_PATH}/{file_name}"

    print(f"Processing file: {file_name}")

    try:
        file_id = calculate_file_hash(file_path)
        print("File ID:", file_id)

        # 1. Check whether this file's data is already present in Bronze
        bronze_file_exists = (
            spark.table(BRONZE_TABLE)
            .filter(F.col("_source_file_id") == file_id)
            .limit(1)
            .count() > 0
        )

        if bronze_file_exists:
            print(f"File already exists in Bronze: {file_name}")

            existing_tracking = (
                spark.table(TRACKING_TABLE)
                .filter(F.col("file_id") == file_id)
                .limit(1)
                .count()
            )

            if existing_tracking == 0:
                print("Repairing missing tracking record...")

                recovery_tracking_df = spark.createDataFrame(
                    [(
                        file_name,
                        file_path,
                        file_modified_time,
                        datetime.now(timezone.utc),
                        None,
                        "SUCCESS",
                        INGESTION_RUN_ID,
                        file_id
                    )],
                    schema="""
                        file_name STRING,
                        file_path STRING,
                        file_modification_time TIMESTAMP,
                        ingestion_timestamp TIMESTAMP,
                        records_ingested BIGINT,
                        ingestion_status STRING,
                        ingestion_run_id STRING,
                        file_id STRING
                    """
                )

                (
                    recovery_tracking_df.write
                    .format("delta")
                    .mode("append")
                    .saveAsTable(TRACKING_TABLE)
                )

            dbutils.fs.mv(file_path, destination_path)
            print(f"Moved file to processed: {file_name}")
            return

        # 2. Read CSV
        source_df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(file_path)
        )

        # 3. Validate schema
        actual_columns = source_df.columns

        missing_columns = [
            col for col in EXPECTED_COLUMNS
            if col not in actual_columns
        ]

        if missing_columns:
            raise ValueError(
                f"Missing required columns: {missing_columns}"
            )

        source_df = source_df.select(*EXPECTED_COLUMNS)

        # 4. Count records
        record_count = source_df.count()

        # 5. Add ingestion metadata
        bronze_input_df = (
            source_df
            .withColumn("_source_file", F.lit(file_name))
            .withColumn("_source_file_id", F.lit(file_id))
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_ingestion_run_id", F.lit(INGESTION_RUN_ID))
        )

        # 6. Append to Bronze
        (
            bronze_input_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(BRONZE_TABLE)
        )

        # 7. Record successful ingestion
        tracking_df = spark.createDataFrame(
            [(
                file_name,
                file_path,
                file_modified_time,
                datetime.now(timezone.utc),
                record_count,
                "SUCCESS",
                INGESTION_RUN_ID,
                file_id
            )],
            schema="""
                file_name STRING,
                file_path STRING,
                file_modification_time TIMESTAMP,
                ingestion_timestamp TIMESTAMP,
                records_ingested BIGINT,
                ingestion_status STRING,
                ingestion_run_id STRING,
                file_id STRING
            """
        )

        (
            tracking_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(TRACKING_TABLE)
        )

        # 8. Move successfully processed file
        dbutils.fs.mv(file_path, destination_path)

        print(
            f"Successfully ingested {record_count} records "
            f"from {file_name}"
        )

    except Exception as e:
        print(f"Failed to ingest {file_name}: {str(e)}")
        print("File remains in incoming for retry.")
        raise

In [0]:
%sql
DESCRIBE TABLE workspace.default.sentinel_bronze_inventory;

col_name,data_type,comment
record_id,bigint,null
product_id,bigint,null
product_name,string,null
category,string,null
supplier_id,string,null
cost_price,double,null
selling_price,double,null
quantity,int,null
reorder_level,int,null
last_updated,date,null


In [0]:
for file_info in dbutils.fs.ls(INCOMING_PATH):
    if not file_info.isDir() and file_info.name.endswith(".csv"):
        ingest_inventory_file(file_info)

Processing file: sentinel_idempotency_test(processing_key_implementation_test).csv
File ID: 26860853af3c010ec21dddafef10a8e0d02a1247add96bf97c4ba1af3fab324c
Successfully ingested 1 records from sentinel_idempotency_test(processing_key_implementation_test).csv


In [0]:
%sql
SELECT
    count(*)
FROM workspace.default.sentinel_bronze_inventory


count(*)
50806


In [0]:
%sql
SELECT *
FROM workspace.default.sentinel_file_tracking;

file_name,file_path,file_modification_time,ingestion_timestamp,records_ingested,ingestion_status,ingestion_run_id,file_id,file_size,error_message
sentinel_idempotency_test(processing_key_implementation_test).csv,dbfs:/Volumes/workspace/default/sentinel_files/inventory/incoming/sentinel_idempotency_test(processing_key_implementation_test).csv,2026-10-31T19:57:13.000Z,2026-10-31T19:57:53.863Z,1,SUCCESS,93c661c2-cf54-434f-9de9-651ab35a882e,26860853af3c010ec21dddafef10a8e0d02a1247add96bf97c4ba1af3fab324c,null,null
sentinel_bronze_inventory_supplement.csv,dbfs:/Volumes/workspace/default/sentinel_files/inventory/incoming/sentinel_bronze_inventory_supplement.csv,2026-10-31T12:20:52.000Z,2026-10-31T12:26:37.566Z,800,SUCCESS,9e66197b-f41b-4d3a-bd2c-afe4d6e6531a,1c8a6d05a2f4046c3e35b1906ddb774e99a50eb461a4e4d74a0aac7a015c12d8,null,null
sentinel_incremental_test.csv,dbfs:/Volumes/workspace/default/sentinel_files/inventory/incoming/sentinel_incremental_test.csv,2026-10-31T17:50:52.000Z,2026-10-31T17:51:19.551Z,5,SUCCESS,2a6dae81-5a3c-4b4d-b9ff-f11ab05ebaa0,cd816c63688e66066f4c3c3db3799b48fd1de0f931e1b3f0583cf4b39b451c24,null,null
sentinel_inventory_test_800.csv,dbfs:/Volumes/workspace/default/sentinel_files/inventory/incoming/sentinel_inventory_test_800.csv,2026-10-31T06:08:25.000Z,2026-10-31T06:09:12.931Z,800,SUCCESS,62e137c9-60e5-4364-9ef9-e4664f180c4e,null,null,null
sentinel_inventory_test.csv,dbfs:/Volumes/workspace/default/sentinel_files/inventory/incoming/sentinel_inventory_test.csv,2026-10-31T06:00:56.000Z,2026-10-31T06:03:48.112Z,10,SUCCESS,62e137c9-60e5-4364-9ef9-e4664f180c4e,null,null,null
